# Генерация аналитических справок по фроду через Ollama

**Перед запуском** — один раз в терминале:
```bash
# 1. Установить Ollama: https://ollama.com/download
# 2. Скачать модель:
ollama pull qwen2.5:7b
# 3. Запустить сервер (если не запущен автоматически):
ollama serve
```
Ollama работает как фоновый HTTP-сервер на `localhost:11434` — Jupyter обращается к нему как к обычному API.

In [ ]:
# ── Установка зависимостей (один раз) ────────────────────────────────────────
import subprocess, sys
subprocess.run([sys.executable, "-m", "pip", "install", "-q",
                "requests", "pandas", "sqlalchemy", "psycopg2-binary", "tqdm"], check=True)
print("OK")

In [ ]:
import json, re, time, logging
import pandas as pd
import requests
from tqdm.notebook import tqdm
from sqlalchemy import create_engine, text

logging.basicConfig(level=logging.WARNING)
log = logging.getLogger(__name__)

## 1. Конфигурация

In [ ]:
# ── Ollama ───────────────────────────────────────────────────────────────────
OLLAMA_MODEL  = "qwen2.5:7b"   
TEMPERATURE   = 0.3            
MAX_RETRIES   = 3

# ── Фильтр ───────────────────────────────────────────────────────────────────
MIN_FRAUD_PROBA = 0.8          
TOP_N_FEATURES  = 5            

## 2. Загрузка данных из БД

In [ ]:
engine = create_engine(DB_URL)

# ── Основная витрина ──────────────────────────────────────────────────────────
dm = pd.read_sql("""
    SELECT
        trans_num,
        trans_date_trans_time,
        category,
        amt,
        hour,
        is_night,
        distance_km,
        client_mean_amt,
        client_tx_count,
        amt_vs_client_mean,
        amt_zscore,
        merchant_fraud_rate,
        fraud_pred_proba
    FROM public."public.fraud_predictions_datamart"
    WHERE fraud_pred = 1
      AND fraud_pred_proba >= %(min_proba)s
""", engine, params={"min_proba": MIN_FRAUD_PROBA})

# ── SHAP-значения (long-формат) ───────────────────────────────────────────────
shap_raw = pd.read_sql("""
    SELECT trans_num, tr_date, fraud_pred_proba, feature, shap, contribution_pct
    FROM public."public.predicted_frauds_shap"
    WHERE trans_num IN (
        SELECT trans_num
        FROM public."public.fraud_predictions_datamart"
        WHERE fraud_pred = 1 AND fraud_pred_proba >= %(min_proba)s
    )
""", engine, params={"min_proba": MIN_FRAUD_PROBA})

print(f"Транзакций к обработке:  {dm['trans_num'].nunique()}")
print(f"Строк SHAP:              {len(shap_raw)}")
dm.head(3)

## 3. Читаемые названия признаков

In [ ]:
FEATURE_LABELS = {
    "amt":                    "сумма транзакции",
    "amt_zscore":             "отклонение суммы от среднего клиента (σ)",
    "amt_vs_client_mean":     "отношение суммы к среднему чеку клиента",
    "amt_vs_merchant_mean":   "отношение суммы к среднему чеку мерчанта",
    "amt_vs_category_mean":   "отношение суммы к среднему по категории",
    "log_amt":                "логарифм суммы транзакции",
    "client_mean_amt":        "средний чек клиента",
    "client_std_amt":         "разброс сумм операций клиента",
    "client_max_amt":         "максимальная операция клиента",
    "client_tx_count":        "количество транзакций клиента в истории",
    "amt_is_max":             "сумма является максимальной для клиента",
    "merchant_fraud_rate":    "доля фрода у данного мерчанта",
    "merchant_mean_amt":      "средний чек мерчанта",
    "merchant_name_len":      "длина названия мерчанта",
    "distance_km":            "расстояние между клиентом и мерчантом (км)",
    "hour":                   "час совершения операции",
    "is_night":               "ночное время операции (23:00–06:00)",
    "is_weekend":             "операция в выходной день",
    "day_of_week":            "день недели",
    "month":                  "месяц проведения операции",
    "age":                    "возраст клиента",
    "age_group":              "возрастная группа клиента",
    "gender_category":        "пол клиента",
    "city_pop":               "численность населения города",
    "lat":                    "широта клиента",
    "long":                   "долгота клиента",
    "merch_lat":              "широта мерчанта",
    "merch_long":             "долгота мерчанта",
    "category":               "категория операции",
}

def human_label(feature: str) -> str:
    return FEATURE_LABELS.get(feature, feature.replace("_", " "))

## 4. Подготовка SHAP: разделение на риск / смягчение

In [ ]:
def split_shap_for_tx(tx_shap: pd.DataFrame, top_n: int = TOP_N_FEATURES):
    """
    Принимает срез shap_raw по одной транзакции.
    Возвращает два списка кортежей (feature, shap, label, contribution_pct):
        risk_factors  — SHAP > 0  (толкают к фроду)
        trust_factors — SHAP < 0  (снижают риск)
    """
    risk  = tx_shap[tx_shap["shap"] > 0].nlargest(top_n, "shap")
    trust = tx_shap[tx_shap["shap"] < 0].nsmallest(top_n, "shap")  # самые отрицательные

    def to_tuples(df):
        return [
            (row["feature"], row["shap"], human_label(row["feature"]),
             row.get("contribution_pct", None))
            for _, row in df.iterrows()
        ]

    return to_tuples(risk), to_tuples(trust)


def format_shap_block(risk, trust) -> str:
    """
    Форматирует оба списка в текстовый блок для промта.
    SHAP > 0 → повышают вероятность фрода
    SHAP < 0 → снижают вероятность фрода
    """
    lines = ["ПРИЗНАКИ, ПОВЫШАЮЩИЕ ВЕРОЯТНОСТЬ ФРОДА (shap > 0):"]
    for feat, shap_val, label, pct in risk:
        pct_str = f", вклад {pct:.1f}%" if pct is not None else ""
        lines.append(f"  + {label}: shap={shap_val:+.4f}{pct_str}")

    if trust:
        lines.append("\nПРИЗНАКИ, СНИЖАЮЩИЕ ВЕРОЯТНОСТЬ ФРОДА (shap < 0):")
        for feat, shap_val, label, pct in trust:
            pct_str = f", вклад {pct:.1f}%" if pct is not None else ""
            lines.append(f"  - {label}: shap={shap_val:+.4f}{pct_str}")
    else:
        lines.append("\nСмягчающих факторов не выявлено.")

    return "\n".join(lines)

## 5. Промт

In [ ]:
def build_prompt(tx_row: pd.Series, tx_shap: pd.DataFrame) -> str:
    risk, trust = split_shap_for_tx(tx_shap)
    shap_block  = format_shap_block(risk, trust)

    # Топ-3 риск-признака войдут в скрипт для клиента
    top3 = ", ".join(label for _, _, label, _ in risk[:3]) or "нетипичные параметры"

    # Читаемое время
    ts = str(tx_row.get("trans_date_trans_time", ""))

    # Контекст клиента
    mean_amt    = tx_row.get("client_mean_amt", "н/д")
    tx_count    = tx_row.get("client_tx_count", "н/д")
    night_flag  = "да" if tx_row.get("is_night", 0) == 1 else "нет"
    dist        = tx_row.get("distance_km", "н/д")
    mfr         = tx_row.get("merchant_fraud_rate", None)
    mfr_str     = f"{mfr:.1%}" if mfr is not None else "н/д"

    mean_str    = f"{mean_amt:,.0f} RUB" if isinstance(mean_amt, float) else str(mean_amt)
    dist_str    = f"{dist:.1f} км" if isinstance(dist, float) else str(dist)

    return f"""Ты — система аналитики безопасности банка. Пиши только на русском языке.

ДАННЫЕ ТРАНЗАКЦИИ:
  ID транзакции:      {tx_row['trans_num']}
  Дата и время:       {ts}
  Сумма:              {tx_row['amt']:,.2f} RUB
  Категория:          {tx_row.get('category', 'н/д')}
  Час операции:       {tx_row.get('hour', 'н/д')}:00
  Ночная операция:    {night_flag}
  Расстояние клиент–мерчант: {dist_str}
  Доля фрода у мерчанта:     {mfr_str}
  Вероятность фрода (модель): {tx_row['fraud_pred_proba']:.1%}

ПРОФИЛЬ КЛИЕНТА:
  Средний чек:        {mean_str}
  Всего операций:     {tx_count}

{shap_block}

ЗАДАЧА: сформируй два текста и верни строго в формате JSON без лишних слов.

1. "operator_report" — справка для оператора колл-центра (150–200 слов):
   - Абзац 1: почему транзакция признана подозрительной (опирайся на признаки с shap > 0,
     сравнивай с профилем клиента)
   - Абзац 2: смягчающие факторы (shap < 0), если есть; если нет — укажи это явно
   - Абзац 3: рекомендация оператору — что уточнить у клиента / как действовать

2. "client_message" — скрипт звонка клиенту (60–80 слов):
   - Нейтральный тон, без слова «мошенничество» и обвинений
   - Назови время ({ts}), сумму ({tx_row['amt']:,.0f} RUB)
   - Упомяни ключевые отличия: {top3}
   - Заверши предложением подтвердить или отклонить операцию

Формат ответа (только JSON, никакого другого текста):
{{"operator_report": "...", "client_message": "..."}}"""

## 6. Вызов Ollama

In [ ]:
OLLAMA_URL = "http://localhost:11434"  # адрес сервера Ollama


def extract_json(raw: str) -> dict:
    """Вытаскивает JSON из ответа модели — даже если она добавила markdown или пояснения."""
    # deepseek-r1 оборачивает рассуждения в <think>...</think> — убираем
    raw = re.sub(r"<think>[\s\S]*?</think>", "", raw).strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        pass
    for pattern in [r"```json\s*([\s\S]+?)\s*```", r"```\s*([\s\S]+?)\s*```", r"(\{[\s\S]+\})"]:
        m = re.search(pattern, raw)
        if m:
            try:
                return json.loads(m.group(1))
            except json.JSONDecodeError:
                continue
    raise ValueError(f"JSON не найден в ответе:\n{raw[:400]}")


def call_ollama(prompt: str) -> dict:
    """Отправляет запрос к Ollama через requests (POST /api/generate)."""
    payload = {
        "model":  OLLAMA_MODEL,
        "prompt": prompt,
        "stream": False,
        "options": {
            "temperature": TEMPERATURE,
            "num_predict": 1200,
        },
    }
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            resp = requests.post(
                f"{OLLAMA_URL}/api/generate",
                json=payload,
                timeout=120,
            )
            resp.raise_for_status()
            raw = resp.json()["response"]
            return extract_json(raw)
        except (ValueError, KeyError) as e:
            # Модель вернула не-JSON — повторяем
            if attempt == MAX_RETRIES:
                raise
            time.sleep(2)
        except requests.RequestException as e:
            # Сетевая ошибка / таймаут
            if attempt == MAX_RETRIES:
                raise
            time.sleep(5)


# ── Проверка соединения ───────────────────────────────────────────────────────
try:
    r = requests.get(f"{OLLAMA_URL}/api/tags", timeout=5)
    r.raise_for_status()
    names = [m["name"] for m in r.json().get("models", [])]
    print("✅ Ollama доступна. Модели:", names)
    if not any(OLLAMA_MODEL in n for n in names):
        print(f"⚠️  Модель {OLLAMA_MODEL} не найдена. Выполни в терминале: ollama pull {OLLAMA_MODEL}")
    else:
        print(f"✅ Модель {OLLAMA_MODEL} готова к работе")
except requests.ConnectionError:
    print("❌ Ollama не отвечает на localhost:11434")
    print("   Запусти в терминале: ollama serve")
except Exception as e:
    print(f"❌ Ошибка: {e}")

## 7. Генерация справок

In [ ]:
# Группируем SHAP по транзакции заранее для скорости
shap_grouped = {tid: grp for tid, grp in shap_raw.groupby("trans_num")}

records = []

for _, tx_row in tqdm(dm.iterrows(), total=len(dm), desc="Генерация справок"):
    tr_num   = tx_row["trans_num"]
    tx_shap  = shap_grouped.get(tr_num, pd.DataFrame())

    if tx_shap.empty:
        records.append({"tr_num": tr_num,
                         "operator_report": "SHAP-данные отсутствуют",
                         "client_message":  "SHAP-данные отсутствуют"})
        continue

    try:
        prompt = build_prompt(tx_row, tx_shap)
        result = call_ollama(prompt)
        records.append({
            "tr_num":          tr_num,
            "operator_report": result.get("operator_report", ""),
            "client_message":  result.get("client_message",  ""),
        })
    except Exception as e:
        records.append({"tr_num": tr_num,
                         "operator_report": f"ОШИБКА: {e}",
                         "client_message":  f"ОШИБКА: {e}"})

result_df = pd.DataFrame(records, columns=["tr_num", "operator_report", "client_message"])
print(f"\nГотово: {len(result_df)} справок")
result_df.head(2)

## 8. Запись в БД

In [ ]:
# Создаём таблицу если нет, затем upsert по tr_num
with engine.begin() as conn:
    conn.execute(text("""
        CREATE TABLE IF NOT EXISTS public.fraud_operator_reports (
            tr_num           TEXT PRIMARY KEY,
            operator_report  TEXT,
            client_message   TEXT,
            generated_at     TIMESTAMP DEFAULT NOW()
        )
    """))

# Upsert: обновляем если запись уже есть
# Передаём только три поля явно — избегаем dict/numpy-типов из других колонок
with engine.begin() as conn:
    for _, row in result_df.iterrows():
        conn.execute(text("""
            INSERT INTO public.fraud_operator_reports (tr_num, operator_report, client_message)
            VALUES (:tr_num, :operator_report, :client_message)
            ON CONFLICT (tr_num) DO UPDATE
                SET operator_report = EXCLUDED.operator_report,
                    client_message  = EXCLUDED.client_message,
                    generated_at    = NOW()
        """), {
            "tr_num":          str(row["tr_num"]),
            "operator_report": str(row["operator_report"]),
            "client_message":  str(row["client_message"]),
        })

print(f"Записано в БД: {len(result_df)} строк → public.fraud_operator_reports")

## 9. Проверка результата

In [ ]:
sample = result_df.iloc[0]
print("=" * 70)
print(f"Транзакция: {sample['tr_num']}")
print("=" * 70)
print("\n📋 СПРАВКА ДЛЯ ОПЕРАТОРА:")
print(sample["operator_report"])
print("\n📞 СКРИПТ ДЛЯ КЛИЕНТА:")
print(sample["client_message"])